# NumPy 快速入门：数组是什么，为什么快？

> 这是「数据分析从入门到精通」系列的第 2 篇。上一篇我们画了整个学习路线图，从这篇开始，正式动手。

---

## 先回答一个问题：为什么不直接用 Python 列表？

学 NumPy 之前，很多人会有一个疑问：

> Python 已经有列表（list）了，可以存数据、可以循环、可以切片——为什么还要专门学 NumPy？

来看一个实验，一眼就懂：

In [ ]:
import numpy as np
import time

# 准备一百万个数字
n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n)

# Python 列表：每个元素乘以 2
start = time.time()
result = [x * 2 for x in py_list]
print(f"Python 列表耗时：{time.time() - start:.4f} 秒")

# NumPy 数组：每个元素乘以 2
start = time.time()
result = np_array * 2
print(f"NumPy 数组耗时：{time.time() - start:.4f} 秒")


**运行结果（参考）：**
```
Python 列表耗时：0.0821 秒
NumPy 数组耗时：0.0009 秒
```

**快了将近 100 倍。**

这还只是一百万个数字。真实的数据分析场景，动辄千万级行，差距只会更大。

---

## NumPy 为什么这么快？

说透了其实就三个原因：

**① 连续内存存储**

Python 列表里的每个元素，是一个独立的 Python 对象，分散存储在内存各处。CPU 读取时要到处"跑"，效率很低。

NumPy 数组把所有元素**紧密排列在一块连续的内存区域**，CPU 可以一次性批量读取，像流水线一样处理。

```
Python 列表（内存分散）：
[obj → 内存A] [obj → 内存B] [obj → 内存C] ...

NumPy 数组（内存连续）：
[1.0][2.0][3.0][4.0][5.0]... （紧密排列）
```

**② 类型统一**

Python 列表可以混装任意类型：`[1, "hello", True, 3.14]`。灵活是灵活，但每次操作都要判断类型，很慢。

NumPy 数组**所有元素类型相同**，比如都是 `float64`。不需要类型判断，运算直接执行。

**③ 向量化运算（底层用 C 实现）**

NumPy 的运算底层是用 **C 语言**写的，绕过了 Python 解释器的开销。`array * 2` 这一行代码，背后调用的是高度优化的 C 函数，直接在 CPU 层面批量处理。

> 简单记：NumPy = 连续内存 + 统一类型 + C 语言底层 → 快。

---

## ndarray：NumPy 的核心对象

NumPy 里最重要的东西只有一个——**ndarray**（n 维数组，n-dimensional array）。

所有操作都围绕它展开。

### 创建一个数组

In [1]:
import numpy as np

# 从 Python 列表创建
a = np.array([1, 2, 3, 4, 5])
print(a)        # [1 2 3 4 5]
print(type(a))  # <class 'numpy.ndarray'>


[1 2 3 4 5]
<class 'numpy.ndarray'>


### 几个最常用的创建方式

In [6]:
# 等差数列（最常用）
np.arange(0, 10, 2)       # [0 2 4 6 8]

array([0, 2, 4, 6, 8])

In [7]:
# 指定个数的等间隔
np.linspace(0, 1, 5)      # [0.   0.25  0.5  0.75  1. ]

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [8]:
# 全零数组
np.zeros((3, 4))           # 3行4列，全是 0.0

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [9]:
# 全一数组
np.ones((2, 3))            # 2行3列，全是 1.0

array([[1., 1., 1.],
       [1., 1., 1.]])

In [11]:
# 随机数组（0到1之间）
np.random.rand(3, 3)       # 3×3 随机矩阵

array([[0.37572891, 0.36871196, 0.80868317],
       [0.35075962, 0.35519865, 0.90924433],
       [0.2476138 , 0.07021287, 0.79909017]])

---

## 数组的两个关键属性

拿到一个数组，先看这两个属性：

In [12]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])

print(a.shape)   # (2, 3) → 2行3列
print(a.dtype)   # int64  → 元素类型


(2, 3)
int64


- **shape**：数组的形状，`(行数, 列数)` 或更高维
- **dtype**：元素的数据类型，常见的有 `int64`、`float64`、`bool`

> 以后遇到报错，80% 的情况都是 shape 不对或 dtype 不对。先检这两个。

---

## 向量化运算：告别 for 循环

NumPy 最重要的使用习惯：**用数组运算代替 for 循环**。

In [ ]:
scores = np.array([72, 85, 91, 63, 78])

# ❌ 旧习惯：for 循环
result = []
for s in scores:
    result.append(s * 1.1)

# ✅ NumPy 方式：直接对数组操作
result = scores * 1.1
# [79.2 93.5 100.1 69.3 85.8]


不只是乘法，加减除、比较、函数——全都可以直接作用在整个数组上：

In [14]:
scores = np.array([72, 85, 91, 63, 78])

scores + 5          # 每个加5

array([77, 90, 96, 68, 83])

In [15]:
scores > 80         # [False  True  True False False]

array([False,  True,  True, False, False])

In [16]:
np.sqrt(scores)     # 每个开根号

array([8.48528137, 9.21954446, 9.53939201, 7.93725393, 8.83176087])

In [17]:
np.mean(scores)     # 均值 → 77.8

np.float64(77.8)

**规则很简单：你对一个数做的事，就对数组里所有数同时做。**

---

## 实战：向量化 vs for 循环，差距有多大？

下面这个场景在数据分析里非常常见——对一列数据做标准化（减均值除以标准差）：

In [18]:
import numpy as np
import time

data = np.random.randn(1_000_000)  # 一百万个数

# 方法一：for 循环
start = time.time()
mean, std = data.mean(), data.std()
result1 = [(x - mean) / std for x in data]
print(f"for 循环耗时：{time.time() - start:.4f} 秒")

# 方法二：NumPy 向量化
start = time.time()
result2 = (data - data.mean()) / data.std()
print(f"NumPy 向量化耗时：{time.time() - start:.4f} 秒")


for 循环耗时：0.1198 秒
NumPy 向量化耗时：0.0031 秒


**参考结果：**
```
for 循环耗时：0.3142 秒
NumPy 向量化耗时：0.0031 秒
```

代码更简洁，速度快 100 倍，结果完全一样。

这就是为什么写数据分析代码时，看到 for 循环就要条件反射地问自己：**这里能不能换成 NumPy？**

---

## 本篇小结

| 概念 | 要点 |
|------|------|
| ndarray | NumPy 的核心数据结构，n 维数组 |
| 为什么快 | 连续内存 + 统一类型 + C 语言底层 |
| shape | 数组的形状，遇到报错先查它 |
| dtype | 元素类型，常用 int64 / float64 |
| 向量化运算 | 对数组直接操作，替代 for 循环 |

NumPy 的内容远不止这些，但掌握这一篇，你已经理解了它最核心的设计思想。接下来的几篇会继续深入：索引切片、广播机制、统计函数——每一个都是实际工作中天天要用的。

---

## 课后练习

试着自己跑一遍下面的代码，观察输出结果：

In [5]:
import numpy as np

# 1. 创建一个 1 到 20 的数组，只保留其中的偶数
a = np.arange(1, 21)
print(a[a % 2 == 0])

# 2. 创建一个 3×3 的随机整数矩阵（0-100之间）
b = np.random.randint(0, 100, size=(3, 3))
print(b)
print("每行最大值：", b.max(axis=1))
print("每列平均值：", b.mean(axis=0))


[ 2  4  6  8 10 12 14 16 18 20]
[[88 91  8]
 [72 12 23]
 [58 23  7]]
每行最大值： [91 72 58]
每列平均值： [72.66666667 42.         12.66666667]


把运行结果截图发到**评论区**，和大家一起对答案 👇

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 3 篇：数组创建、索引与切片**
>
> 有了数组之后，怎么精准取出你想要的那一行、那一列、那个区间？下篇用一份**考试成绩表**带你掌握 NumPy 索引的全部核心用法。

---

**觉得这篇讲清楚了？**

👇 点个 **「在看」**，让更多想学数据分析的人看到  
💬 评论区告诉我：运行代码后，你的机器上快了多少倍？  
⭐ 关注公众号，每周 3 篇，跟着节奏学不迷路

---

*「数据分析从入门到精通」系列 · 第 2 篇*  
*上一篇：[第 1 篇：5阶段数据分析完整路线图]*  
*下一篇：第 3 篇：数组创建、索引与切片*